# MediSummarize: handwritten word OCR experiment
Compare Tesseract, pretrained TrOCR-small, and TrOCR-small fine-tuned on RxHandBD v3.

**This notebook has been prepared, not GPU-executed.** It does not establish full-page handwriting recognition or clinical correctness.

1. In Kaggle create a notebook, enable Internet and a GPU, then import this notebook.
2. Run installation once and restart the kernel before continuing.
3. Download the official ML archive below, or attach your own copy as a private Kaggle dataset and set `ARCHIVE`.
4. Audit first. Open `review-50.html`, review the 50 training labels, and inspect the eight near-duplicate flags. Record decisions before training.
5. Run validation baselines; train for three epochs, selecting the checkpoint by validation CER.
6. Freeze settings, then enable the final test cell once. Download all output files, including predictions, audit, environment, and the model checkpoint.

Dataset: Md Masudul Islam, RxHandBD v3, DOI [10.17632/dsb5r6vskg.3](https://data.mendeley.com/datasets/dsb5r6vskg/3), CC BY 4.0. Model: [Microsoft TrOCR-small handwritten](https://huggingface.co/microsoft/trocr-small-handwritten), already fine-tuned on IAM. Keep their attribution and model license with exported weights. No API token or commercial model service is required.


In [ ]:
%pip install "transformers==4.57.6" "accelerate==1.12.0" "sentencepiece==0.2.1" "pytesseract==0.3.13" "Pillow>=11,<13"
!apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-eng


Restart the kernel after installation. Kaggle supplies PyTorch; do not replace its CUDA build unnecessarily.

In [ ]:
from pathlib import Path
import hashlib, json, subprocess, sys, urllib.request
WORK = Path('/kaggle/working/medisummarize-ocr')
WORK.mkdir(parents=True, exist_ok=True)
ARCHIVE = WORK / 'RxHandBD-ML.zip'
URL = 'https://data.mendeley.com/public-files/datasets/dsb5r6vskg/files/98acd360-ba4c-45d0-b614-5d66f4deee47/file_downloaded'
if not ARCHIVE.exists():
    urllib.request.urlretrieve(URL, ARCHIVE)
expected = 'c6371c0a3a89301c0b6b2eac1d2d49f6f3d8e559a7a71b338655dde0a6a63b51'
assert hashlib.file_digest(ARCHIVE.open('rb'), 'sha256').hexdigest() == expected, 'Wrong archive/version'
print('Verified official RxHandBD v3 ML archive')


In [ ]:
# Write the same audited modules kept in the repository.
MODULES = {'metrics.py': '"""Corpus metrics shared by the local baseline and Kaggle experiments."""\nimport re\nimport unicodedata\n\n\ndef normalize(text):\n    return " ".join(unicodedata.normalize("NFC", str(text)).split())\n\n\ndef distance(a, b):\n    previous = list(range(len(b) + 1))\n    for i, left in enumerate(a, 1):\n        current = [i]\n        for j, right in enumerate(b, 1):\n            current.append(min(current[-1] + 1, previous[j] + 1, previous[j - 1] + (left != right)))\n        previous = current\n    return previous[-1]\n\n\ndef score(pairs):\n    """Case-sensitive corpus CER/WER; lower is better. Values may exceed 1."""\n    rows = [(normalize(ref), normalize(pred)) for ref, pred in pairs]\n    if not rows or any(not ref for ref, _ in rows):\n        raise ValueError("Metrics require nonempty reference labels and predictions for every sample.")\n    digit_rows = [(r, p) for r, p in rows if re.search(r"\\d", r)]\n    return {\n        "samples": len(rows),\n        "cer": sum(distance(r, p) for r, p in rows) / sum(len(r) for r, _ in rows),\n        "wer": sum(distance(r.split(), p.split()) for r, p in rows) / sum(len(r.split()) for r, _ in rows),\n        "exact_match": sum(r == p for r, p in rows) / len(rows),\n        "digit_samples": len(digit_rows),\n        "digit_sequence_match": (sum(re.findall(r"\\d+", r) == re.findall(r"\\d+", p) for r, p in digit_rows) / len(digit_rows)) if digit_rows else None,\n        "normalization": "Unicode NFC, whitespace collapsed, case preserved",\n    }\n', 'audit.py': '"""Audit official RxHandBD v3 ML data without silently repairing labels.\n\npython experiments/handwriting/audit.py --archive data/runtime/rxhandbd-v3/RxHandBD-ML.zip --out data/runtime/rxhandbd-v3/audit\n"""\nfrom __future__ import annotations\nimport argparse\nimport base64\nimport csv\nimport hashlib\nimport html\nimport io\nimport json\nimport random\nimport zipfile\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nfrom PIL import Image, ImageOps\n\nEXPECTED_SHA256 = "c6371c0a3a89301c0b6b2eac1d2d49f6f3d8e559a7a71b338655dde0a6a63b51"\n\n\ndef audit(archive: Path, out: Path):\n    out.mkdir(parents=True, exist_ok=True)\n    archive_hash = hashlib.file_digest(archive.open("rb"), "sha256").hexdigest()\n    if archive_hash != EXPECTED_SHA256:\n        raise ValueError("Archive differs from the official v3 ML file. Do not mix dataset versions.")\n    rows, issues, thumbnails = [], [], {}\n    with zipfile.ZipFile(archive) as z:\n        names = set(z.namelist())\n        for split, label_file, image_dir in [("train", "Train_Label.csv", "Train_Set"), ("test", "Test_Label.csv", "Test_Set")]:\n            prefix = "RxHandBD-ML/"\n            reader = csv.DictReader(io.StringIO(z.read(prefix + label_file).decode("utf-8-sig"), newline=None))\n            for item in reader:\n                filename, label = item["Images"].strip(), item["Text"]\n                name = prefix + image_dir + "/" + filename\n                if not label.strip() or name not in names:\n                    issues.append({"split": split, "file": filename, "reason": "missing image or empty label"})\n                    continue\n                raw = z.read(name)\n                try:\n                    im = Image.open(io.BytesIO(raw)).convert("RGB")\n                    im.load()\n                except Exception as exc:\n                    issues.append({"split": split, "file": filename, "reason": str(exc)})\n                    continue\n                gray = ImageOps.grayscale(im).resize((9, 8))\n                pixels = list(gray.getdata())\n                bits = [pixels[y * 9 + x] > pixels[y * 9 + x + 1] for y in range(8) for x in range(8)]\n                dhash = sum(int(bit) << i for i, bit in enumerate(bits))\n                pixel_hash = hashlib.sha256(str(im.size).encode() + im.tobytes()).hexdigest()\n                rows.append({"path": name, "filename": filename, "text": label, "official_split": split,\n                             "width": im.width, "height": im.height, "sha256": hashlib.sha256(raw).hexdigest(),\n                             "pixel_sha256": pixel_hash, "dhash": f"{dhash:016x}"})\n                if split == "train":\n                    # Contact sheet uses train images only; test labels never drive label repair.\n                    thumb = im.copy(); thumb.thumbnail((220, 140))\n                    buf = io.BytesIO(); thumb.save(buf, format="PNG")\n                    thumbnails[name] = base64.b64encode(buf.getvalue()).decode()\n        image_names = {n for n in names if n.lower().endswith((".jpg", ".jpeg", ".png"))}\n    groups = defaultdict(list)\n    for row in rows:\n        groups[row["pixel_sha256"]].append(row)\n    overlaps = [items for items in groups.values() if len({r["official_split"] for r in items}) > 1]\n    conflicts = [items for items in groups.values() if len({r["text"] for r in items}) > 1]\n    # Preserve the official test set; exclude exact duplicate test content from fitting and validation.\n    test_hashes = {r["pixel_sha256"] for r in rows if r["official_split"] == "test"}\n    eligible_groups = sorted(h for h, items in groups.items() if h not in test_hashes and items[0]["official_split"] == "train")\n    random.Random(1019).shuffle(eligible_groups)\n    val_hashes = set(eligible_groups[:max(1, round(len(eligible_groups) * 0.1))])\n    conflicting_hashes = {items[0]["pixel_sha256"] for items in conflicts}\n    for row in rows:\n        h = row["pixel_sha256"]\n        row["split"] = "test" if row["official_split"] == "test" else "excluded" if h in test_hashes or h in conflicting_hashes else "validation" if h in val_hashes else "train"\n    train_candidates = [r for r in rows if r["split"] == "train"]\n    eval_candidates = [r for r in rows if r["split"] in {"validation", "test"}]\n    # dHash is only a triage flag: collisions do NOT prove duplication or trigger removal.\n    near = []\n    for left in train_candidates:\n        lh = int(left["dhash"], 16)\n        for right in eval_candidates:\n            if left["pixel_sha256"] != right["pixel_sha256"] and (lh ^ int(right["dhash"], 16)).bit_count() <= 2:\n                near.append({"train": left["filename"], "evaluation": right["filename"], "split": right["split"]})\n    label_names = [r["path"] for r in rows]\n    report = {\n        "dataset": "RxHandBD v3 ML", "doi": "10.17632/dsb5r6vskg.3", "license": "CC BY 4.0",\n        "source": "https://data.mendeley.com/datasets/dsb5r6vskg/3", "archive_sha256": archive_hash,\n        "official_counts": dict(Counter(r["official_split"] for r in rows)),\n        "experiment_counts": dict(Counter(r["split"] for r in rows)),\n        "unique_labels": len({r["text"] for r in rows}), "image_shapes": dict(Counter(f\'{r["width"]}x{r["height"]}\' for r in rows)),\n        "unlabeled_images": sorted(image_names - set(label_names)),\n        "repeated_label_rows": len(label_names) - len(set(label_names)), "invalid_items": issues,\n        "exact_duplicate_groups": sum(len(v) > 1 for v in groups.values()),\n        "cross_official_split_duplicate_groups": len(overlaps), "conflicting_label_groups": len(conflicts),\n        "near_duplicate_flags": len(near),\n        "split_seed": 1019, "validation_fraction_groups": 0.1,\n        "writer_independent": False, "writer_metadata": "Not supplied in archive labels; prescription/writer leakage cannot be ruled out.",\n        "limitations": ["Word crops only; no full-page layout recognition benchmark.", "Near-duplicate flags need human inspection; they are not established duplicates.", "No writer-independent claim is supported.", "Official test set retained; exact test duplicates excluded from fitting/validation."]\n    }\n    for name, payload in [("audit.json", report), ("duplicate-groups.json", [[{k:r[k] for k in ("path","text","official_split")} for r in v] for v in groups.values() if len(v)>1]), ("near-duplicate-flags.json", near)]:\n        (out / name).write_text(json.dumps(payload, indent=2), encoding="utf-8")\n    with (out / "manifest.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=list(rows[0])); writer.writeheader(); writer.writerows(rows)\n    sample = random.Random(1019).sample(train_candidates, min(50, len(train_candidates)))\n    contact = "<!doctype html><meta charset=\'utf-8\'><title>RxHandBD training-label review</title><style>body{font:16px system-ui;margin:32px}section{display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:20px}article{border:1px solid #aaa;padding:12px}img{max-width:100%;height:140px;object-fit:contain}label{display:block}</style><h1>Review 50 training labels</h1><p>Mark unreadable crops, wrong labels, ambiguous digits, and unexpected language in manual-review.csv. Do not silently correct test labels.</p><section>"\n    for r in sample:\n        contact += f\'<article><img alt="Handwritten training crop" src="data:image/png;base64,{thumbnails[r["path"]]}"><h2>{html.escape(r["text"])}</h2><p>{html.escape(r["filename"])}</p></article>\'\n    (out / "review-50.html").write_text(contact + "</section>", encoding="utf-8")\n    with (out / "manual-review.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.writer(f); writer.writerow(["filename", "provided_label", "decision", "notes"])\n        writer.writerows((r["filename"], r["text"], "pending", "") for r in sample)\n    print(json.dumps(report, indent=2))\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(); parser.add_argument("--archive", type=Path, required=True); parser.add_argument("--out", type=Path, required=True)\n    args = parser.parse_args(); audit(args.archive, args.out)\n', 'run_experiment.py': '"""Run cropped-word OCR experiments locally or in Kaggle. No cloud API key.\n\nCommands: audit first, baseline (validation), train (validation selection), test (held-out once).\n"""\nfrom __future__ import annotations\nimport argparse\nimport csv\nimport hashlib\nimport io\nimport json\nimport platform\nimport random\nimport time\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nfrom PIL import Image\nfrom metrics import score\n\nMODEL = "microsoft/trocr-small-handwritten"\n\n\ndef read_rows(path, split):\n    with path.open(encoding="utf-8", newline="") as f:\n        return [r for r in csv.DictReader(f) if r["split"] == split]\n\n\ndef image_from(archive, row):\n    return Image.open(io.BytesIO(archive.read(row["path"]))).convert("RGB")\n\n\ndef load_trocr(checkpoint, device):\n    from transformers import TrOCRProcessor, VisionEncoderDecoderModel\n    processor = TrOCRProcessor.from_pretrained(checkpoint)\n    model = VisionEncoderDecoderModel.from_pretrained(checkpoint).to(device)\n    model.config.decoder_start_token_id = processor.tokenizer.cls_token_id\n    model.config.pad_token_id = processor.tokenizer.pad_token_id\n    model.config.eos_token_id = processor.tokenizer.sep_token_id\n    model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id\n    model.generation_config.pad_token_id = processor.tokenizer.pad_token_id\n    model.generation_config.eos_token_id = processor.tokenizer.sep_token_id\n    return processor, model\n\n\ndef predict_trocr(processor, model, archive, rows, device, batch_size):\n    import torch\n    predictions = []\n    model.eval()\n    with torch.inference_mode():\n        for start in range(0, len(rows), batch_size):\n            images = [image_from(archive, row) for row in rows[start:start+batch_size]]\n            values = processor(images=images, return_tensors="pt").pixel_values.to(device)\n            tokens = model.generate(values, max_new_tokens=64, num_beams=1)\n            predictions.extend(processor.batch_decode(tokens, skip_special_tokens=True))\n    return predictions\n\n\ndef evaluate(name, archive, rows, out, split, predictor, metadata):\n    destination = out / f"{name}-{split}.json"\n    if destination.exists():\n        raise FileExistsError(f"{destination} already exists. Keep the recorded result; do not overwrite test evidence.")\n    started = time.perf_counter(); predictions = predictor()\n    if len(predictions) != len(rows):\n        raise ValueError("A prediction is required for every sample, including OCR failures.")\n    seconds = time.perf_counter() - started\n    metrics = score([(r["text"], p) for r, p in zip(rows, predictions)])\n    payload = {\n        "schema_version": 1, "task": "handwriting_word_ocr", "model": name,\n        "dataset": "RxHandBD v3 ML", "dataset_sha256": metadata["archive_sha256"],\n        "split": split, "samples": len(rows), "metrics": metrics, "elapsed_seconds": seconds,\n        "created_at": datetime.now(timezone.utc).isoformat(), "is_mock": False,\n        "provenance": metadata, "limitations": ["Cropped words, not whole handwritten pages.", "No verified writer-independent split.", "Imported results require researcher review."],\n    }\n    destination.write_text(json.dumps(payload, indent=2), encoding="utf-8")\n    with (out / f"{name}-{split}-predictions.csv").open("w", newline="", encoding="utf-8") as f:\n        writer = csv.writer(f); writer.writerow(["filename", "reference", "prediction", "exact_match"])\n        writer.writerows((r["filename"], r["text"], p, r["text"] == p) for r, p in zip(rows, predictions))\n    print(name, json.dumps(metrics))\n    return payload\n\n\ndef train(args, archive, audit):\n    import torch\n    from torch.utils.data import DataLoader, Dataset\n    random.seed(1019); torch.manual_seed(1019)\n    if torch.cuda.is_available(): torch.cuda.manual_seed_all(1019)\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    if device == "cpu" and not args.allow_cpu:\n        raise RuntimeError("Enable Kaggle GPU first. CPU training requires explicit --allow-cpu.")\n    processor, model = load_trocr(MODEL, device)\n    train_rows = read_rows(args.manifest, "train")\n    validation = read_rows(args.manifest, "validation")\n    if not train_rows or not validation:\n        raise ValueError("Nonempty train and validation splits are required.")\n    # Fail rather than silently truncating medical labels.\n    if any(len(processor.tokenizer(r["text"]).input_ids) > 64 for r in train_rows + validation):\n        raise ValueError("A label exceeds 64 tokens. Review lengths before changing this protocol.")\n    class Words(Dataset):\n        def __len__(self): return len(train_rows)\n        def __getitem__(self, index):\n            row = train_rows[index]\n            pixels = processor(images=image_from(archive, row), return_tensors="pt").pixel_values.squeeze(0)\n            labels = processor.tokenizer(row["text"], padding="max_length", max_length=64, return_tensors="pt").input_ids.squeeze(0)\n            labels[labels == processor.tokenizer.pad_token_id] = -100\n            return {"pixel_values": pixels, "labels": labels}\n    loader = DataLoader(Words(), batch_size=args.batch_size, shuffle=True, num_workers=0)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)\n    model.gradient_checkpointing_enable()\n    model.config.use_cache = False\n    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")\n    best, history = float("inf"), []\n    checkpoint = args.out / "best-checkpoint"\n    if checkpoint.exists(): raise FileExistsError("Use a new output directory for another training run.")\n    configuration = {\n        "checkpoint": MODEL, "revision": getattr(model.config, "_commit_hash", None),\n        "archive_sha256": audit["archive_sha256"],\n        "manifest_sha256": hashlib.file_digest(args.manifest.open("rb"), "sha256").hexdigest(),\n        "seed": 1019, "epochs": args.epochs, "batch_size": args.batch_size,\n        "gradient_accumulation": args.accumulation, "learning_rate": 5e-5,\n        "weight_decay": 0.01, "selection_metric": "validation CER", "device": device,\n        "train_samples": len(train_rows), "validation_samples": len(validation),\n    }\n    (args.out / "training-config.json").write_text(json.dumps(configuration, indent=2), encoding="utf-8")\n    for epoch in range(args.epochs):\n        model.train(); optimizer.zero_grad(); total_loss = 0.0\n        for step, batch in enumerate(loader):\n            batch = {key: value.to(device) for key, value in batch.items()}\n            with torch.autocast(device_type=device, enabled=device == "cuda"):\n                loss = model(**batch).loss\n            if not torch.isfinite(loss): raise ValueError("Non-finite training loss; stop and inspect.")\n            # Correct scaling for the final, possibly partial accumulation group.\n            group_size = min(args.accumulation, len(loader) - (step // args.accumulation) * args.accumulation)\n            scaler.scale(loss / group_size).backward(); total_loss += loss.item()\n            if (step + 1) % args.accumulation == 0 or step + 1 == len(loader):\n                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                scaler.step(optimizer); scaler.update(); optimizer.zero_grad()\n        predictions = predict_trocr(processor, model, archive, validation, device, args.batch_size)\n        validation_metrics = score([(r["text"], p) for r, p in zip(validation, predictions)])\n        history.append({"epoch": epoch + 1, "train_loss": total_loss / len(loader), "validation": validation_metrics})\n        print(json.dumps(history[-1]))\n        if validation_metrics["cer"] < best:\n            best = validation_metrics["cer"]\n            model.save_pretrained(checkpoint); processor.save_pretrained(checkpoint)\n    (args.out / "training-history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")\n    print("Saved best validation checkpoint:", checkpoint)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--phase", choices=["baseline", "train", "test"], required=True)\n    parser.add_argument("--archive", type=Path, required=True); parser.add_argument("--manifest", type=Path, required=True)\n    parser.add_argument("--out", type=Path, required=True); parser.add_argument("--engine", choices=["tesseract", "trocr", "all"], default="all")\n    parser.add_argument("--batch-size", type=int, default=4); parser.add_argument("--accumulation", type=int, default=4)\n    parser.add_argument("--epochs", type=int, default=3); parser.add_argument("--allow-cpu", action="store_true")\n    args = parser.parse_args()\n    if min(args.batch_size, args.accumulation, args.epochs) < 1: parser.error("Batch size, accumulation and epochs must be positive.")\n    args.out.mkdir(parents=True, exist_ok=True)\n    audit = json.loads((args.manifest.parent / "audit.json").read_text(encoding="utf-8"))\n    if hashlib.file_digest(args.archive.open("rb"), "sha256").hexdigest() != audit["archive_sha256"]:\n        raise ValueError("Dataset changed after audit.")\n    if audit["invalid_items"] or audit["repeated_label_rows"] or audit["unlabeled_images"]:\n        raise ValueError("Resolve invalid/missing/repeated label rows before running experiments.")\n    manifest_sha = hashlib.file_digest(args.manifest.open("rb"), "sha256").hexdigest()\n    with zipfile.ZipFile(args.archive) as archive:\n        if args.phase == "train": return train(args, archive, audit)\n        split = "validation" if args.phase == "baseline" else "test"\n        rows = read_rows(args.manifest, split)\n        metadata = {"archive_sha256": audit["archive_sha256"], "manifest_sha256": manifest_sha, "seed": 1019, "python": platform.python_version(), "normalization": "NFC + whitespace; case preserved", "tesseract_psm": 7, "decoder": "greedy; max_new_tokens=64", "writer_independent": False}\n        if args.engine in {"tesseract", "all"}:\n            import pytesseract\n            metadata["tesseract_version"] = str(pytesseract.get_tesseract_version())\n            def tesseract():\n                # An empty OCR prediction still counts as an error; exceptions abort the run.\n                return [pytesseract.image_to_string(image_from(archive, r), config="--psm 7", lang="eng").strip() for r in rows]\n            evaluate("Tesseract", archive, rows, args.out, split, tesseract, metadata)\n        if args.engine in {"trocr", "all"}:\n            import torch, transformers\n            device = "cuda" if torch.cuda.is_available() else "cpu"\n            metadata.update(torch=torch.__version__, transformers=transformers.__version__, device=torch.cuda.get_device_name(0) if device == "cuda" else "CPU")\n            for label, checkpoint in [("Pretrained TrOCR-small", MODEL)] + ([("Fine-tuned TrOCR-small", str(args.out / "best-checkpoint"))] if args.phase == "test" else []):\n                if label.startswith("Fine-tuned"):\n                    training = json.loads((args.out / "training-config.json").read_text(encoding="utf-8"))\n                    if training["manifest_sha256"] != manifest_sha or training["archive_sha256"] != audit["archive_sha256"]:\n                        raise ValueError("Training and evaluation must use the same frozen archive and split manifest.")\n                    metadata["training"] = training\n                processor, model = load_trocr(checkpoint, device)\n                metadata["checkpoint"] = checkpoint\n                metadata["checkpoint_revision"] = getattr(model.config, "_commit_hash", None)\n                evaluate(label, archive, rows, args.out, split, lambda: predict_trocr(processor, model, archive, rows, device, args.batch_size), metadata.copy())\n                del model\n                if device == "cuda": torch.cuda.empty_cache()\n\n\nif __name__ == "__main__": main()\n'}
for name, source in MODULES.items():
    (WORK / name).write_text(source, encoding='utf-8')


In [ ]:
import torch, transformers
print('GPU available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before training'
environment = {'torch': torch.__version__, 'transformers': transformers.__version__, 'gpu': torch.cuda.get_device_name(0)}
(WORK / 'environment.json').write_text(json.dumps(environment, indent=2))
(WORK / 'pip-freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
AUDIT = WORK / 'audit'
subprocess.run([sys.executable, str(WORK/'audit.py'), '--archive', str(ARCHIVE), '--out', str(AUDIT)], check=True)


Open the generated HTML and CSV from Kaggle’s Output files. Inspect `near-duplicate-flags.json` against the corresponding images before freezing the split. Similarity hashes can collide; do not remove samples just because a flag exists. Any split revision must be documented and happen before final testing.

In [ ]:
MANUAL_REVIEW_DONE = False  # Set True only after inspecting the label sample and near-duplicate flags.
assert MANUAL_REVIEW_DONE, 'Complete the manual dataset review first'
OUT = WORK / 'run-01'
BASE = [sys.executable, str(WORK/'run_experiment.py'), '--archive', str(ARCHIVE), '--manifest', str(AUDIT/'manifest.csv'), '--out', str(OUT)]
subprocess.run(BASE + ['--phase', 'baseline'], check=True)


In [ ]:
# Start with batch size 4 and effective batch size 16. On out-of-memory, use batch size 2 and accumulation 8.
# Keep the same final settings for all scored runs; record any exploratory changes.
subprocess.run(BASE + ['--phase', 'train', '--epochs', '3', '--batch-size', '4', '--accumulation', '4'], check=True)


Review training-history.json and validation predictions. Freeze preprocessing, decoding, and training settings. Do not choose settings by test performance. The final cell evaluates all three systems on the same held-out official test images; the script refuses to overwrite existing result files.

In [ ]:
RUN_FINAL_TEST = False  # Enable once after settings are frozen; do not use test scores to tune the model.
if RUN_FINAL_TEST:
    subprocess.run(BASE + ['--phase', 'test'], check=True)
else:
    print('Final test not run. Validation results are not final test results.')


In [ ]:
# Export only aggregate result JSON to the app. Keep raw predictions for your report/error review.
results = [json.loads(p.read_text()) for p in OUT.glob('*-test.json')]
if results:
    (WORK/'experiment-results.json').write_text(json.dumps({'schema_version': 1, 'runs': results}, indent=2))
    print('Download experiment-results.json plus the entire run-01 and audit folders.')
else:
    print('No final test results to export yet.')
